<a href="https://colab.research.google.com/github/navsingh2024-bit/Gen_AI-Property_price_prediction-/blob/main/Final_Gen_AI_House_Price_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛠️ Phase 0: Environment Stabilization & Setup
**Objective:** Prepare the runtime environment.
We install all required dependencies here. Pinning `scikit-learn` to version `1.3.2` is critical to prevent version mismatch errors later during deployment. We also install the LangChain, Groq, LangGraph, and HuggingFace libraries needed for our 100% free AI agent framework.

In [ ]:
# Phase 1: Environment Setup & Data Ingestion
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Scikit-Learn Modules
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn import metrics

# Load the proprietary dataset
df = pd.read_csv('gurgaon_properties_post_feature_selection_v2.csv')

print("--- Data Ingestion Complete ---")
print(f"Total Properties: {df.shape[0]}")
print(f"Total Features: {df.shape[1]}")

# 1. Statistical Summary
print("--- Summary Statistics ---")
display(df.describe())

# 2. Identify Numeric Columns for Correlation
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns

# 3. Feature Correlation Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='Blues', fmt=".2f", annot_kws={"size": 10})
plt.title("Numeric Feature Correlation Matrix (Gurgaon Properties)")
plt.show()

# 📊 Phase 1: Data Ingestion & Exploratory Data Analysis (EDA)
**Objective:** Load and understand the data.
In this phase, we load the cleaned proprietary real estate dataset. We print out its shape, display summary statistics, and visualize the numeric correlations using a Seaborn heatmap to understand data relationships before modeling.

In [ ]:
# Phase 1: Environment Setup & Data Ingestion
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Scikit-Learn Modules
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn import metrics

# Load the proprietary dataset
df = pd.read_csv('gurgaon_properties_post_feature_selection_v2.csv')

print("--- Data Ingestion Complete ---")
print(f"Total Properties: {df.shape[0]}")
print(f"Total Features: {df.shape[1]}")

# 1. Statistical Summary
print("--- Summary Statistics ---")
display(df.describe())

# 2. Identify Numeric Columns for Correlation
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns

# 3. Feature Correlation Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='Blues', fmt=".2f", annot_kws={"size": 10})
plt.title("Numeric Feature Correlation Matrix (Gurgaon Properties)")
plt.show()

# 🧠 Phase 2: Model Architecture & Pipeline Construction
**Objective:** Build a robust, leak-proof Machine Learning pipeline.
We split the data into features (X) and target (y). To process the data cleanly, we build a `ColumnTransformer` to handle scaling (for numerical data) and One-Hot Encoding (for categorical data). Finally, we bundle this processor with a `RandomForestRegressor` into a single `Pipeline` and train it.

In [ ]:
# Phase 2: Model Architecture & Scikit-Learn Pipeline

# 1. Separate Features (X) and Target (y)
X = df.drop(columns=['price'])
y = df['price']

# 2. Dynamically identify categorical vs numerical columns
numeric_features = X.select_dtypes(include=['float64', 'int64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

# 3. Build the Preprocessing Engine
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# 4. Construct the Final Pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('rf_model', RandomForestRegressor(
        n_estimators=100,
        max_depth=15,
        min_samples_split=5,
        random_state=42,
        n_jobs=-1 # Uses all cores for fast training
    ))
])

# 5. Train/Test Split & Execution
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Initiating Pipeline Training on Gurgaon Dataset...")
pipeline.fit(X_train, y_train)
print("Model Training Complete.")

# 📈 Phase 3: Model Evaluation & Serialization
**Objective:** Test the model and save it for future use.
We evaluate our pipeline against unseen test data, calculating core metrics: R-Squared, Mean Absolute Error (MAE), and Root Mean Squared Error (RMSE). We then plot the Actual vs. Predicted prices to visualize accuracy. Finally, the model is serialized into a `.pkl` file so the AI agent can use it later.

In [ ]:
# Phase 3: Model Evaluation & Saving

# Generate predictions on unseen test data
y_pred = pipeline.predict(X_test)

# Calculate Core Metrics
r2 = metrics.r2_score(y_test, y_pred)
mae = metrics.mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(metrics.mean_squared_error(y_test, y_pred))

print('--- Model Performance on Test Set ---')
print(f'R-Squared (Variance Explained): {r2:.4f}')
print(f'Mean Absolute Error (MAE):      {mae:.4f} Crores')
print(f'Root Mean Squared Error (RMSE): {rmse:.4f} Crores')

# Visualizing Actual vs. Predicted Prices
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.4, color='teal')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel("Actual Property Price (Crores)")
plt.ylabel("Predicted Property Price (Crores)")
plt.title("Actual vs. Predicted Prices (Gurgaon Properties)")
plt.show()

# Save the model for UI/Agent integration
model_filename = 'rf_gurgaon_pipeline_final.pkl'
joblib.dump(pipeline, model_filename, compress=3)

print(f"Pipeline successfully serialized and saved as '{model_filename}'")

# 📚 Phase 4: RAG Implementation (Vector Database Setup)
**Objective:** Build a searchable Knowledge Base for the AI.
Here, we prepare the actual historical dataset for the LLM. We convert every row in our dataframe into a descriptive text paragraph. Then, we use free **HuggingFace Embeddings** (`all-MiniLM-L6-v2`) to turn these texts into vectors and store them in a FAISS database. Finally, we expose this database as a searchable "Tool" for our LangGraph Agent.

In [ ]:
!pip install langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers

In [ ]:
# Phase 4: RAG Setup with Free HuggingFace Embeddings
from langchain_community.document_loaders import DataFrameLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# --- FIX: Updated import for modern LangChain versions ---
from langchain_core.tools import create_retriever_tool

print("Building Knowledge Base for RAG...")

# 1. Create a descriptive text column for the Vector Store
df_rag = df.copy()
df_rag['rag_content'] = df_rag.apply(
    lambda row: f"{row['bedRoom']} BHK {row['property_type']} in {row['sector']}. "
                f"Price: {row['price']} Crores. Area: {row['built_up_area']} sqft. "
                f"Age: {row['agePossession']}. Luxury Category: {row['luxury_category']}.",
    axis=1
)

# 2. Load documents using DataFrameLoader
loader = DataFrameLoader(df_rag, page_content_column="rag_content")
docs = loader.load()

# 3. Create FAISS Vector Store using 100% Free HuggingFace Embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(docs, embeddings)

# 4. Create a Retriever Tool for LangGraph
retriever = vector_store.as_retriever(search_kwargs={"k": 4})
rag_tool = create_retriever_tool(
    retriever,
    "search_actual_properties",
    "Searches the real estate database for actual properties. Use this when the user asks for examples, existing listings, or real data."
)

print("RAG Vector Store and Tool created successfully!")

# ⚙️ Phase 5: Machine Learning Tool Wrapping
**Objective:** Give the AI the ability to make mathematical predictions.
We write a Python function that loads the saved Random Forest `.pkl` model to generate price estimates for hypothetical properties. By decorating this function with `@tool`, we allow the Groq LLM to automatically understand its parameters and "call" it when a user asks for a price prediction.

In [ ]:
# Phase 5: Machine Learning Tool for LangGraph
from langchain_core.tools import tool
import pandas as pd
import joblib

@tool
def predict_property_price(property_type: str, sector: str, bedRoom: str, bathroom: str,
                           balcony: str, agePossession: str, built_up_area: str,
                           servant_room: str, store_room: str, furnishing_type: str,
                           luxury_category: str, floor_category: str) -> str:
    """
    Predicts the price of a hypothetical property in Gurgaon.
    """
    try:
        model = joblib.load('rf_gurgaon_pipeline_final.pkl')

        input_data = pd.DataFrame([{
            'property_type': property_type,
            'sector': sector,
            'bedRoom': float(bedRoom),
            'bathroom': float(bathroom),
            'balcony': balcony,
            'agePossession': agePossession,
            'built_up_area': float(built_up_area),
            'servant room': float(servant_room),
            'store room': float(store_room),
            'furnishing_type': furnishing_type,
            'luxury_category': luxury_category,
            'floor_category': floor_category
        }])

        prediction = model.predict(input_data)[0]
        return f"Model Prediction: The estimated price for this {bedRoom} BHK in {sector} is {prediction:.2f} Crores."

    except Exception as e:
        # --- FIX: Print the error to the Colab console so we can debug it! ---
        error_msg = f"Error running the prediction model: {str(e)}."
        print(f"\n⚠️ INTERNAL TOOL ERROR: {error_msg}\n")
        return error_msg

print("ML Prediction Tool generated successfully!")

# 🤖 Phase 6: LangGraph ReAct Agent Setup (Groq API)
**Objective:** Compile the AI brain.
We securely load the Groq API Key from Google Colab's Secrets manager. Then, we initialize `ChatGroq` using the high-performance `llama-3.1-70b-versatile` model (which is excellent at tool calling). We use LangGraph to compile a ReAct agent, giving it access to both our tools: the **RAG Tool** (for real data) and the **ML Tool** (for predictions).

In [ ]:
!pip install langchain-groq

In [ ]:
# Phase 6: Building the LangGraph Agent with GROQ
import os
from google.colab import userdata
from langgraph.prebuilt import create_react_agent
from langchain_groq import ChatGroq

try:
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
except userdata.SecretNotFoundError:
    print("❌ ERROR: 'GROQ_API_KEY' not found in Colab Secrets.")

print("Compiling LangGraph Agent...")

# --- FIX: Switching to 8B model temporarily to bypass the Rate Limit ---
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

tools = [rag_tool, predict_property_price]
agent_executor = create_react_agent(llm, tools=tools)

print("LangGraph Agent compiled and ready!")

# 🚀 Phase 7: Execution & Testing
**Objective:** Chat with our Real Estate AI.
This cell contains a simple execution function to feed messages to our compiled LangGraph Agent. It streams the responses back to us and prints out whenever the Agent autonomously decides to trigger one of our tools. We will run two tests to verify it knows the difference between asking for *actual data* vs *hypothetical predictions*.

In [ ]:
# Phase 7: Execution & Testing Function
def chat_with_agent(user_query: str):
    print(f"\n--- USER: {user_query} ---")

    inputs = {"messages": [("user", user_query)]}

    # --- FIX: Added a strict limit of 5 tool calls to prevent infinite loops! ---
    config = {"recursion_limit": 5}

    try:
        for chunk in agent_executor.stream(inputs, stream_mode="values", config=config):
            message = chunk["messages"][-1]

            if message.type == "ai" and message.content:
                print(f"🤖 AGENT: {message.content}")
            elif message.type == "ai" and getattr(message, 'tool_calls', None):
                for tool_call in message.tool_calls:
                    print(f"🛠️  [Agent triggered tool: {tool_call['name']} ...]")
    except Exception as e:
        print(f"🛑 Execution stopped: {str(e)}")

# TEST 1: RAG
chat_with_agent("Can you show me a few real examples of 3 BHK flats in sector 57 from our database?")

# TEST 2: ML Pipeline
chat_with_agent(
    "Estimate the price of a flat in sector 53. It has 3 bedrooms, 4 bathrooms, 3+ balconies, "
    "is an Old Property, 2600 sq ft, 0 servant rooms, 0 store rooms. It's Medium luxury, low floor. "
    "Furnishing is 0.0."
)